<a href="https://colab.research.google.com/github/matthewpecsok/IS4490-creation-fall2026/blob/main/module-02-evaluating-language-models-for-business-tasks/module_02_assignment_02_local_model_task_portfolio_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ### Note on Labs and Assignments:
>
> 🔧 Look for the **wrench emoji** — it marks code you must change. Routine run-only cells do not use it.
>
> 🖊 Look for the **writing emoji** — it marks analysis you must write.
>
> These sections are graded and are not optional.
>

### TODO - IDENTIFY YOURSELF 🔧

In [ ]:
STUDENT_NAME = "Your Name" # TODO REPLACE YOUR NAME HERE 🔧

# Module 2 Assignment 2: Local Model Task Portfolio

**Notebook:** Student Template  
**Student:** Edit the configuration cell  
**Required models:** `gemma3:1b`, `gemma3:4b`, `llama3.2:1b`, and `llama3.2:3b`

This notebook supports the complete Assignment 2 workflow: four direct business
tasks, prompt revision, a controlled four-model comparison, evidence-based scoring, reflection, and AI-use disclosure.


## Student Introduction: What is this Assignment Is About?

This assignment asks you to work directly with small language models running on your own machine via Ollama. You will practice two core skills:

1. **Prompt engineering** — writing and iteratively improving instructions that guide a model toward a useful business output.
2. **Model evaluation** — systematically comparing four models on the same task and scoring them with evidence.

**What you will produce:**

- **Part 1:** Four business tasks (extraction, summarization, drafting, classification). For each task, you write an initial zero-shot instruction, run it, diagnose a specific weakness in the output, revise the instruction using a named prompting strategy, run it again, and evaluate the improvement.
- **Part 2:** A controlled four-model comparison using a shared service-request packet. You design a single instruction, run it on all four models without changing anything, then score each model across four dimensions with evidence from their outputs.
- **Part 3:** A 350–500 word reflection answering seven specific questions about what you observed.

**Before you start:**
1. Run the initial code blocks to install Ollama and start it.
2. Replace `"Your Name"` in the configuration cell below with your actual name.
3. Run all cells from top to bottom in order.

The notebook will raise an error and stop if any required `TODO` is still present when you try to run a model — this is intentional so you do not accidentally submit incomplete work.

## Important Instructions

1. Read the assignment before editing this notebook.
2. Edit only cells marked for student work.
3. Do not change the comparison source packet, model list, or shared settings.
4. Preserve the first output from every run.
5. Before submitting, restart the kernel and run all cells from top to bottom.

The template intentionally raises a clear error when a required `TODO` remains.


## Setup Ollama

This notebook will download, install and start [Ollama](https://ollama.com/download). The four default model downloads require approximately 8 GB in total.





### What is Ollama?

Ollama is a tool that lets you run AI language models on your own computer instead of only using an online service like ChatGPT.

For a beginner, you can think of it as a local “AI model manager.” It helps you download a model, start it, and send it prompts. For example, instead of calling an online API from OpenAI, Google, or Anthropic, you can call an Ollama model running on your laptop or server.

The basic idea is:



*   You install Ollama.
*   You download a model, such as Llama, Gemma, or Mistral.
*   You send text to the model.
*  The model sends text back.


Why are we doing this rather than using Claude or ChatGPT?

* Reproducability. The notebook allows students to all follow the same steps and instructions.
* Cost: API access to Anthropic,OpenAI, Google models is not free. These models are free to run. The trade-off? (There always is one).
* What do we sacrifice for using the free models? Quality of responses and speed. We'll be using CPUs since these models are small language models (SLMs) rather than large language models (LLMs)






In [ ]:
import subprocess
import time

In [ ]:
# Download and install Ollama (Google Colab only — skip if running locally)
install_zstd = subprocess.run(
    "sudo apt-get install zstd",
    shell=True,
    capture_output=True,
    text=True,
)

install_zstd

CompletedProcess(args='sudo apt-get install zstd', returncode=0, stdout='Reading package lists...\nBuilding dependency tree...\nReading state information...\nThe following NEW packages will be installed:\n  zstd\n0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.\nNeed to get 603 kB of archives.\nAfter this operation, 1,695 kB of additional disk space will be used.\nGet:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]\nFetched 603 kB in 0s (11.9 MB/s)\nSelecting previously unselected package zstd.\n(Reading database ... \n(Reading database ... 5%\n(Reading database ... 10%\n(Reading database ... 15%\n(Reading database ... 20%\n(Reading database ... 25%\n(Reading database ... 30%\n(Reading database ... 35%\n(Reading database ... 40%\n(Reading database ... 45%\n(Reading database ... 50%\n(Reading database ... 55%\n(Reading database ... 60%\n(Reading database ... 65%\n(Reading database ... 70%\n(Reading database ... 75%\n(Reading data

In [ ]:
# RUN THIS CELL. YOU SHOULD SEE Ollama installed and Ollama server is running messages.

# Download and install Ollama
install = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    raise RuntimeError(f"Ollama installation failed:\n{install.stderr}")
print("Ollama installed.")

# Start the Ollama server as a background process
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Give the server a few seconds to initialize before any requests are made
time.sleep(3)
print("Ollama server is running.")

Ollama installed.
Ollama server is running.


In [ ]:
# RUN THIS CELL

from datetime import datetime
from hashlib import sha256
from time import perf_counter
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import textwrap

from IPython.display import Markdown, display
import subprocess
import time


REFERENCE_MODE = False
AUTO_PULL_MODELS = True
OLLAMA_BASE_URL = "http://localhost:11434"

REQUIRED_MODELS = [
    "gemma3:1b",
    "gemma3:4b",
    "llama3.2:1b",
    "llama3.2:3b",
]
BASELINE_MODEL = "gemma3:1b"
GENERATION_OPTIONS = {
    "temperature": 1,
    #"seed": 4490,
    "num_ctx": 8192,
    "num_predict": 900,
}

if STUDENT_NAME == "Your Name":
    print("TODO: Replace STUDENT_NAME before submitting.")

print(f"Student: {STUDENT_NAME}")
print(f"Reference mode: {REFERENCE_MODE}")
print(f"Required models: {', '.join(REQUIRED_MODELS)}")
print(f"Shared settings: {GENERATION_OPTIONS}")


TODO: Replace STUDENT_NAME before submitting.
Student: Your Name
Reference mode: False
Required models: gemma3:1b, gemma3:4b, llama3.2:1b, llama3.2:3b
Shared settings: {'temperature': 1, 'num_ctx': 8192, 'num_predict': 900}


In [ ]:
# RUN THIS CELL

REFERENCE_OUTPUTS = {}


def require_finished(label, value):
    """Stop before a model run when a required student field is unfinished."""
    if value is None or "TODO" in str(value):
        raise ValueError(f"Complete {label} before running this cell.")


def ollama_request(path, payload=None, timeout=120):
    """Send a JSON request to the local Ollama service."""
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{OLLAMA_BASE_URL}{path}",
        data=data,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        details = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Ollama returned HTTP {exc.code}: {details}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(
            "Cannot connect to Ollama at http://localhost:11434. "
            "Install and start Ollama, then rerun this cell."
        ) from exc


def chat_once(model, prompt, run_key):
    """Run one independent prompt and return content plus observable metadata."""
    if REFERENCE_MODE:
        fixture = REFERENCE_OUTPUTS[run_key]
        return {
            "model": model,
            "run_key": run_key,
            "recorded_at": fixture["recorded_at"],
            "elapsed_seconds": fixture["elapsed_seconds"],
            "content": fixture["content"],
            "prompt_eval_count": None,
            "eval_count": None,
            "reference_fixture": True,
        }

    started_at = datetime.now().astimezone().isoformat(timespec="seconds")
    start = perf_counter()
    response = ollama_request(
        "/api/chat",
        {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "keep_alive": 0,
            "options": GENERATION_OPTIONS,
        },
        timeout=900,
    )

    #print(f"Generation Options:{GENERATION_OPTIONS}")

    elapsed = round(perf_counter() - start, 2)
    return {
        "model": model,
        "run_key": run_key,
        "recorded_at": started_at,
        "elapsed_seconds": elapsed,
        "content": response["message"]["content"].strip(),
        "prompt_eval_count": response.get("prompt_eval_count"),
        "eval_count": response.get("eval_count"),
        "reference_fixture": False,
    }


def display_record(record):
    metadata = (
        f"**Model:** `{record['model']}`  \n"
        f"**Recorded:** {record['recorded_at']}  \n"
        f"**Elapsed:** {record['elapsed_seconds']} seconds"
    )
    display(Markdown(metadata))
    display(Markdown(record["content"]))


def compose_prompt(instruction, source):
    return (
        instruction.strip()
        + "\n\nSOURCE\n------\n"
        + source.strip()
    )


def run_part1(task_key, stage, instruction, source):
    require_finished(f"{task_key} {stage} instruction", instruction)
    return chat_once(
        BASELINE_MODEL,
        compose_prompt(instruction, source),
        f"part1_{task_key}_{stage}",
    )


In [ ]:
# RUN THIS CELL
# You should see
# Attempting to pull model: gemma3:1b
# Successfully pulled gemma3:1b.
# etc.

# Check and pull required models if AUTO_PULL_MODELS is True
if AUTO_PULL_MODELS:
    print(f"Checking and pulling required models: {REQUIRED_MODELS}")
    for model_name in REQUIRED_MODELS:
        print(f"Attempting to pull model: {model_name}")
        # Use subprocess to run ollama pull command
        pull_result = subprocess.run(
            f"ollama pull {model_name}",
            shell=True,
            capture_output=True,
            text=True,
        )
        if pull_result.returncode != 0:
            print(f"Failed to pull {model_name}:\n{pull_result.stderr}")
        else:
            print(f"Successfully pulled {model_name}.")
        time.sleep(1) # Give a moment between pulls

Checking and pulling required models: ['gemma3:1b', 'gemma3:4b', 'llama3.2:1b', 'llama3.2:3b']
Attempting to pull model: gemma3:1b
Successfully pulled gemma3:1b.
Attempting to pull model: gemma3:4b
Successfully pulled gemma3:4b.
Attempting to pull model: llama3.2:1b
Successfully pulled llama3.2:1b.
Attempting to pull model: llama3.2:3b
Successfully pulled llama3.2:3b.


# Part 1: Direct AI Task Portfolio

In this part you complete four independent business tasks using the baseline model (`gemma3:1b`). Each task gives you a realistic business scenario and a source text. Your job is to engineer the prompt that produces the most useful output.

**How each task section works:**

1. **Write a zero-shot instruction** in the first code cell. Zero-shot means plain directions only — no examples, no step-by-step reasoning prompts. The code comment already labels it for you.
2. **Run the cell and preserve the output.** Do not delete or re-run the initial output before recording it. Your submission must show the original output.
3. **Diagnose the output** in the markdown cell that follows. Compare what the model produced against the source text. Identify one specific, evidence-based weakness (something missing, wrong, or poorly formatted). Quote or paraphrase both the source and the output.
4. **Write a revised instruction** in the second code cell, addressing the weakness you identified. For **at least two of the four tasks**, apply a named prompting strategy and identify it at the top of your instruction string.
5. **Evaluate the revision** in the final markdown cell. Explain whether the revision materially improved the output, what role the chosen strategy played, and what decisions still require a human's judgment.

**Named strategies you may apply for revised instructions:**

| Strategy | What it means |
|---|---|
| **Few-shot** | Include one or more examples of the expected input/output pattern before the task. |
| **Chain-of-thought** | Instruct the model to reason step by step before giving its final answer. |
| **Persona** | Assign the model a specific role or professional background before the task. |
| **Zero-shot** | Plain directions only — acceptable when the initial output already meets your standards. |

Preserve every initial output before revising an instruction.

In [ ]:
PART1_SOURCES = {
  "extraction": "From: Maya Chen\nTo: Facilities Service Desk\nSubject: Loose handrail before Friday tour\n\nThe handrail in the east stairwell on floor 3 of Pioneer Hall is loose at the\nlower wall bracket. I noticed it at 9:15 a.m. on September 2. No one has been\ninjured, and the stairwell is still open. Please repair it before the visitor\ntour this Friday if possible. I can meet a technician after 1:00 p.m. Call me\nat extension 5521.",
  "summarization": "Customer Elena Ruiz reported that order OR-8841 was charged twice. The\noriginal $186.40 charge posted on August 6, and a second $186.40 charge posted\non August 8 after she refreshed the checkout page. The order itself arrived on\nAugust 10 and was correct. Agent Malik opened case CS-2197 on August 11 and\nasked Billing to investigate. Billing has not yet confirmed whether the second\nentry is a settled charge or a temporary authorization. Elena wants the second\ncharge removed if it settled, but she does not want the order canceled. She\nasked for an update by August 13 because her card payment is due August 14.",
  "drafting": "Supplier Northstar Filtration notified Procurement that shipment NF-771,\ncontaining 12 replacement filters, will arrive August 19 instead of August 14.\nThe plant currently has approximately four days of filter inventory at normal\nusage. Northstar offered expedited shipping for an additional fee, but\nProcurement has not approved that option. Operations is checking whether usage\ncan be reduced safely. The plant manager needs a status update today. No\nproduction shutdown has been scheduled.",
  "classification": "Routing categories:\n- IT Support: computers, software, networks, and accounts\n- Security Access: badges, controlled doors, and physical-access permissions\n- Facilities: building fixtures, utilities, and room conditions\n\nUrgency rules:\n- Urgent: an active safety issue or current business operation is blocked with\n  no workaround\n- Standard: future need, routine repair, or a workable temporary alternative\n\nRequest: \"My new analyst starts Monday. Her employee account works, but her\nbadge does not open the Finance Annex. I can meet her in the lobby and escort\nher on the first day if needed. Please add normal weekday access before 8:00\na.m. Monday. The request does not include the analyst's employee ID or the\nmanager's access approval record.\" "
}


## Part 1.1: Extraction

In this section you are fulfilling the role of an AI Automation Specialist working as a consultant for a local facilities management operation. They have been struggling with the amount of time it takes to read and synthesize information from unstructured emails from customers at various facilities. Your job is to extract key items from emails.

Your first prompt must be a **ZERO-SHOT PROMPT**. In the next section you will use other strategies to improve the output.

**Business user:** Facilities coordinator  
**Purpose:** Turn an emailed repair request into a consistent intake record.

**Provided input**

```text
From: Maya Chen
To: Facilities Service Desk
Subject: Loose handrail before Friday tour

The handrail in the east stairwell on floor 3 of Pioneer Hall is loose at the
lower wall bracket. I noticed it at 9:15 a.m. on September 2. No one has been
injured, and the stairwell is still open. Please repair it before the visitor
tour this Friday if possible. I can meet a technician after 1:00 p.m. Call me
at extension 5521.
```


#### TODO - INSTRUCT 🔧

In [ ]:
# 🔧 TODO - INSTRUCT
# Strategy: zero-shot — clear directions only, no examples.
# Your GOAL here is to extract the following pieces of information from the email.
# Extract reporter name, location, problem description, date and time observed, urgency or deadline, and contact information."
initial_instruction_extraction = """Extract reporter name, location, problem description, date and time observed, urgency or deadline, and contact information."""
#initial_instruction_extraction = """TODO REPLACE WITH ZERO SHOT PROMPT"""

In [ ]:
# Run this to generate output after updating your instruction. It will take at least 20 seconds to run each loop.

for i in range(3):

  initial_record_extraction = run_part1(
      task_key="extraction",
      stage="initial",
      instruction=initial_instruction_extraction,
      source=PART1_SOURCES["extraction"],
  )
  display_record(initial_record_extraction)

**Model:** `gemma3:1b`  
**Recorded:** 2026-06-18T18:24:19+00:00  
**Elapsed:** 59.79 seconds

Here’s the extracted information based on the text provided:

*   **Reporter Name:** Maya Chen (explicitly stated)
*   **Location:** Pioneer Hall, East stairwell (floor 3)
*  **Problem Description:** Loose handrail
*   **Date Observed & Time:** September 2nd at approximately 9:15 a.m.
*   **Urgency/Deadline:** Immediate – Repair before Friday's visitor tour.
*   **Contact Information:** Extension 5521

**Model:** `gemma3:1b`  
**Recorded:** 2026-06-18T18:25:19+00:00  
**Elapsed:** 3.98 seconds

Here's an extraction of information from the provided text, formatted as requested:

*   **Reporter Name:** Maya Chen
*   **Location:** Pioneer Hall (east stairwell, floor 3)
    *   Specific location within Pioneer Hall: "stairwell on floor 3"
*   **Problem Description:** Loose handrail at lower wall bracket.
*  **Date & Time of Observation:** September 2nd, 9:15 a.m. (approximate time)
*   **Urgency/Deadline:** Repair before the visitor tour this Friday (if possible). Contact information provided.

**Model:** `gemma3:1b`  
**Recorded:** 2026-06-18T18:25:23+00:00  
**Elapsed:** 3.95 seconds

Here’s a breakdown of the information extracted from Maya Chen's report:

* **Reporter:** Maya Chen
* **Location:** Pioneer Hall (east stairwell, floor 3) - specifically at the lower wall bracket
*  **Problem Description:** The handrail in the east stairwell is loose.
*   **Date and Time of Observation:** September 2nd @ 9:15 a.m.
* **Urgency/Deadline:** Required to be repaired before the visitor tour this Friday (potentially)
* **Contact Information:** Extension 5521

### TODO - REFLECT 🖊

Initial-Output Diagnosis and Revision Plan



**Specific weakness in the initial output:**

🖊 TODO: Cite evidence from the source and the initial output before writing the revised instruction.

**Speed of Output:**

🖊 TODO: How long (on average, in seconds) did the model take to produce the outputs when using a CPU?

🖊 TODO: How long (on average, in seconds) did the model take to produce the outputs when using a GPU?

**Planned instruction change:**

🖊 TODO: State how the revised instruction will address the diagnosed weakness.

**Prompting strategy for the revision:**

🖊 TODO: Name the strategy you will apply — zero-shot, few-shot, chain-of-thought, or persona — and explain why it suits this task and weakness.

### Revise your prompt with a strategy to address the issue you noted above with the model output quality.

### TODO - INSTRUCT 🔧

In [ ]:
# 🔧 TODO. Write the revised instruction.
revised_instruction_extraction = """TODO: Write a revised instruction using a strategy"""

revised_instruction_extraction = """

Extract reporter name, location, problem description, date and time observed, urgency or deadline, and contact information.

example template to follow:
* Reporter Name: NAME
* Location: PHYSICAL LOCATION DETAILS
* Problem Description: DESCRIBE THE PROBLEM
* Date & Time Observed: MM/DD/YY HH:MM
* Urgency / Deadline: DESCRIBE THE URGENCY OF THE ISSUE
* Contact Information: HOW TO REACH THE REPORTING INDIVIDUAL

"""

In [ ]:
# run this to create the output. It will take at least 20 seconds to run each loop.
for i in range(3):

  revised_record_extraction = run_part1(
      task_key="extraction",
      stage="revised",
      instruction=revised_instruction_extraction,
      source=PART1_SOURCES["extraction"],
  )
  display_record(revised_record_extraction)

**Model:** `gemma3:1b`  
**Recorded:** 2026-06-18T18:25:27+00:00  
**Elapsed:** 4.05 seconds

Okay, here's an analysis of the provided template data extracted from the source text:

* **Reporter Name:** Maya Chen
* **Location:** Pioneer Hall (specifically East stairwell on floor 3)
* **Problem Description:** Loose handrail in the east stairwell - lower wall bracket.
* **Date & Time Observed:** 09/21/2024, 09:15AM PST
* **Urgency / Deadline:** PRIORITIZING – Need to repair this before Friday’s visitor tour.
* **Contact Information (Reporting Individual):** Extension 5521 - Maya Chen


This response incorporates the information from the template to accurately identify and summarize the details in the source text.

**Model:** `gemma3:1b`  
**Recorded:** 2026-06-18T18:25:31+00:00  
**Elapsed:** 3.99 seconds

Here's an extraction based on your provided text template, presented as requested:

*   **Reporter Name:** MAYA Chen
*   **Location:** Pioneer Hall – east stairwell on floor 3
*   **Problem Description:** Loose handrail at lower wall bracket in the eastern stairs.
* **Date & Time Observed**: September 2nd, 9:15 a.m.
*  **Urgency / Deadline:** Immediate action required to repair before the Friday visitor tour. Report to extension 5521 for technician availability.
*   **Contact Information:** Extension 5521

**Model:** `gemma3:1b`  
**Recorded:** 2026-06-18T18:25:35+00:00  
**Elapsed:** 4.01 seconds

* Reporter Name: MAYA CHENG
* Location: Pioneer Hall (East Stairwell, Floor 3)
* Problem Description: Loose handrail on lower wall bracket in the east stairwell. The issue was detected at 9:15 a.m. September 2nd and is currently open.
* Date & Time Observed: 09/02/2024 09:15 AM
* Urgency / Deadline: REQUIRED - Repair before visitor tour (Friday)  - Ideally by 1 PM after technician meeting.
* Contact Information: MAYA CHEN – Extension 5521

### TODO - REFLECT 🖊

Improvement and Human Review

**Effect of the revision:**

🖊TODO: Explain whether the revision materially improved the output and why.

**Role of the prompting strategy:**

🖊TODO: Explain what the named strategy contributed — or, if you stayed with zero-shot, explain why no examples, reasoning steps, or persona were needed.

**Human review still required:**

🖊TODO: Identify decisions, facts, risks, or actions that remain a person's responsibility.

**Output Variability**

🖊TODO: How much did the model output change from run to run? Based on your observations, how many of the 3 outputs could be used effectively?

## Part 1.2: Summarization

**Business user:** Customer-service supervisor  
**Purpose:** Prepare a concise escalation summary without losing financial details.

**Provided input**

```text
Customer Elena Ruiz reported that order OR-8841 was charged twice. The
original $186.40 charge posted on August 6, and a second $186.40 charge posted
on August 8 after she refreshed the checkout page. The order itself arrived on
August 10 and was correct. Agent Malik opened case CS-2197 on August 11 and
asked Billing to investigate. Billing has not yet confirmed whether the second
entry is a settled charge or a temporary authorization. Elena wants the second
charge removed if it settled, but she does not want the order canceled. She
asked for an update by August 13 because her card payment is due August 14.
```


**Your task:** A customer-service supervisor needs a concise escalation summary to hand off to a billing team. The model should condense the case above without losing any financially important detail — amounts, dates, case numbers, and the customer's stated deadline all matter.

**What to do in the cells below:**
- **First code cell:** Replace the `TODO` with your zero-shot instruction. Specify the audience (the billing team), the required level of detail, and any format constraints (length, structure). Run the cell and leave the output visible.
- **Diagnosis markdown cell:** Compare the output against the source. Identify one specific weakness — for example, a missing amount, a dropped date, or a format that buries the urgency.
- **Second code cell:** Write your revised instruction addressing that weakness. Remember: at least two of your four tasks must apply a named strategy with an explanation.
- **Evaluation markdown cell:** Explain what improved, what the strategy contributed, and which facts in the summary would need human verification before the billing team acts on them.

### TODO - INSTRUCT 🔧

In [ ]:
# 🔧 Strategy: zero-shot — clear directions only, no examples.
initial_instruction_summarization = """TODO: Write your first (zero-shot) instruction — clear directions, no examples."""

In [ ]:
for i in range(3):

  initial_record_summarization = run_part1(
      task_key="summarization",
      stage="initial",
      instruction=initial_instruction_summarization,
      source=PART1_SOURCES["summarization"],
  )
  display_record(initial_record_summarization)

ValueError: Complete summarization initial instruction before running this cell.

### TODO - REFLECT 🖊

Initial-Output Diagnosis and Revision Plan

Use Strategy: zero-shot — clear directions only, no examples.

**Specific weakness in the initial output:**

🖊TODO: What weaknesses did the output show?




**Planned instruction change:**

🖊TODO: State how the revised instruction will address the diagnosed weakness.

**Prompting strategy for the revision:**

🖊TODO: Name the strategy you will apply — zero-shot, few-shot, chain-of-thought, or persona — and explain why it suits this task and weakness.

**Your task:** A procurement analyst needs to send the plant manager a status update about a delayed shipment. The draft must be factually accurate, avoid making commitments that have not been approved (e.g., expedited shipping has not been authorized), and convey appropriate urgency without overstating the risk.

**What to do in the cells below:**
- **First code cell:** Replace the `TODO` with your zero-shot instruction. Specify the intended recipient (the plant manager), the tone, and any constraints on what the draft should or should not commit to. Run the cell and leave the output visible.
- **Diagnosis markdown cell:** Review the draft for any facts that differ from the source, commitments the model made that are not supported, or tone and structure problems.
- **Second code cell:** Write your revised instruction. Consider whether a persona strategy (e.g., "You are a procurement analyst...") or chain-of-thought reasoning helps the model avoid unsupported commitments.
- **Evaluation markdown cell:** Explain what improved, what the strategy contributed, and what a human analyst must check before sending the draft.

### TODO - INSTRUCT 🔧

In [ ]:
# 🔧 Identify your strategy at the start of the instruction string, e.g.: "Strategy: persona"
revised_instruction_summarization = """TODO: Write a revised instruction. If applying a named strategy, identify it at the start (e.g., "Strategy: persona")."""

In [ ]:
for i in range(3):

  revised_record_summarization = run_part1(
      task_key="summarization",
      stage="revised",
      instruction=revised_instruction_summarization,
      source=PART1_SOURCES["summarization"],
  )
  display_record(revised_record_summarization)

### TODO - REFLECT 🖊

Improvement and Human Review

**Effect of the revision:**

🖊 TODO: Explain whether the revision materially improved the output and why.

**Role of the prompting strategy:**

🖊 TODO: Explain what the named strategy contributed — or, if you stayed with zero-shot, explain why no examples, reasoning steps, or persona were needed.

**Human review still required:**

🖊 TODO: Identify decisions, facts, risks, or actions that remain a person's responsibility.

## Part 1.3: Drafting

**Business user:** Procurement analyst  
**Purpose:** Draft an internal delay notice that does not make unsupported commitments.

**Provided input**

```text
Supplier Northstar Filtration notified Procurement that shipment NF-771,
containing 12 replacement filters, will arrive August 19 instead of August 14.
The plant currently has approximately four days of filter inventory at normal
usage. Northstar offered expedited shipping for an additional fee, but
Procurement has not approved that option. Operations is checking whether usage
can be reduced safely. The plant manager needs a status update today. No
production shutdown has been scheduled.
```


**Your task:** A service-desk dispatcher needs to route this access request to the correct team and assign the correct urgency level. The routing and urgency definitions are included in the source text above — the model should apply them, not invent its own categories. The model should also flag that required information (employee ID and approval record) is missing.

**What to do in the cells below:**
- **First code cell:** Replace the `TODO` with your zero-shot instruction. Tell the model to output the routing category, urgency level, and any missing information that blocks processing, based strictly on the definitions provided. Run the cell and leave the output visible.
- **Diagnosis markdown cell:** Check whether the model applied the urgency definitions correctly (does the workaround change the urgency?), used only the defined categories, and flagged the missing fields. Identify the most significant gap.
- **Second code cell:** Write your revised instruction. Chain-of-thought is often effective here — prompting the model to reason through each definition before giving a final answer can reduce misclassification.
- **Evaluation markdown cell:** Explain what improved, what the strategy contributed, and what a human dispatcher must still decide before acting on the model's output.

### TODO - INSTRUCT 🔧

In [ ]:
# 🔧 TODO - INSTRUCT
# Strategy: zero-shot — clear directions only, no examples.
initial_instruction_drafting = """TODO: Write your first (zero-shot) instruction — clear directions, no examples."""

In [ ]:
# Run this to generate output after updating your instruction. It will take at least 20 seconds to run each loop.

for i in range(3):
  initial_record_drafting = run_part1(
      task_key="drafting",
      stage="initial",
      instruction=initial_instruction_drafting,
      source=PART1_SOURCES["drafting"],
  )
  display_record(initial_record_drafting)

### TODO - REFLECT 🖊

Initial-Output Diagnosis and Revision Plan

**Specific weakness in the initial output:**

🖊 TODO: Cite evidence from the source and the initial output before writing the revised instruction.

**Planned instruction change:**

🖊 TODO: State how the revised instruction will address the diagnosed weakness.

**Prompting strategy for the revision:**

🖊 TODO: Name the strategy you will apply — zero-shot, few-shot, chain-of-thought, or persona — and explain why it suits this task and weakness.

### TODO - INSTRUCT 🔧

In [ ]:
# 🔧 TODO - INSTRUCT
# Identify your strategy at the start of the instruction string, e.g.: "Strategy: chain-of-thought"
revised_instruction_drafting = """TODO: Write a revised instruction. If applying a named strategy, identify it at the start (e.g., "Strategy: chain-of-thought")."""

In [ ]:
# Run this to generate output after updating your instruction.

revised_record_drafting = run_part1(
    task_key="drafting",
    stage="revised",
    instruction=revised_instruction_drafting,
    source=PART1_SOURCES["drafting"],
)
display_record(revised_record_drafting)

### TODO - REFLECT 🖊

Improvement and Human Review

**Effect of the revision:**

🖊 TODO: Explain whether the revision materially improved the output and why.

**Role of the prompting strategy:**

🖊 TODO: Explain what the named strategy contributed — or, if you stayed with zero-shot, explain why no examples, reasoning steps, or persona were needed.

**Human review still required:**

🖊 TODO: Identify decisions, facts, risks, or actions that remain a person's responsibility.

# Part 2: Controlled Four-Model Evaluation

In Part 1, you were free to revise your instructions and iterate. Part 2 is different: you write **one** instruction and send it to all four models **unchanged**. This is a controlled experiment — the only variable is the model itself.

**Why controlled?** If you give different prompts to different models, any differences in output could come from your instruction, not from the model. A shared, unmodified prompt isolates the model as the only variable and makes your comparisons valid.



## Service Issue Details

```text
REQUEST SR-2401
Site: North Distribution Center
Reported by: Luis Ortega, extension 4410
At 6:40 a.m. on June 12, the quality-control freezer display read 18 F. Its
required operating range is 0-5 F. Temperature-sensitive calibration
materials were moved to the backup freezer. Staff reset the alarm twice,
but it returned both times. No employee injury was reported.
```


In [ ]:
multimodeltext = """
REQUEST SR-2401
Site: North Distribution Center
Reported by: Luis Ortega, extension 4410
At 6:40 a.m. on June 12, the quality-control freezer display read 18 F. Its
required operating range is 0-5 F. Temperature-sensitive calibration
materials were moved to the backup freezer. Staff reset the alarm twice,
but it returned both times. No employee injury was reported.
"""

### TODO - INSTRUCT 🔧

In [ ]:
# 🔧 TODO - INSTRUCT
# Strategy: zero-shot — clear directions only, no examples.
initial_instruction_extraction = """TODO: Use your updated summarization instruction from Part 1.1 here"""

initial_instruction_extraction = """

Extract reporter name, location, problem description, date and time observed, urgency or deadline, and contact information.

example template to follow:
* Reporter Name: NAME
* Location: PHYSICAL LOCATION DETAILS
* Problem Description: DESCRIBE THE PROBLEM
* Date & Time Observed: MM/DD/YY HH:MM
* Urgency / Deadline: DESCRIBE THE URGENCY OF THE ISSUE
* Contact Information: HOW TO REACH THE REPORTING INDIVIDUAL

"""

In [ ]:
# Run this after revising the

comparison_prompt = compose_prompt(
    initial_instruction_extraction,
    multimodeltext,
)

comparison_records = {}
for model in REQUIRED_MODELS:
    print(f"Running independent comparison: {model}")
    comparison_records[model] = chat_once(
        model,
        comparison_prompt,
        f"comparison_{model}",
    )
    display_record(comparison_records[model])


### TODO - REFLECT 🖊

Model Evaluation

**How similar or different were the outputs?:**

🖊 TODO: Cite evidence from the output.

**How long did each model take to run?:**

🖊 TODO: How fast or slow were each of the models? Do you notice a pattern for models that ran faster vs slower?

**Which model was "Best"?:**

🖊 TODO: Are the model differences substantial enough to merit choosing a particular model? If so, which?

# Part 3: Reflect on the Assignment



#### TODO - FINAL REFLECTION 🖊

Write a **350–500 word** reflection and answer the

1. **Prompt effect (Part 1):** Which revision across your four Part 1 tasks produced the largest improvement? What specifically changed in the model's output, and why did the revised instruction work better?

🖊 TODO: YOUR ANSWER HERE

2. **Strategy fit:** For each named strategy you applied (few-shot, chain-of-thought, or persona), explain why you chose it for that task. If you kept zero-shot for any revision, explain why plain directions were sufficient.

🖊 TODO: YOUR ANSWER HERE

3. **Model differences (Part 2):** Where did the four models diverge most noticeably — in accuracy, completeness, format adherence, or something else? Give a specific example.

🖊 TODO: YOUR ANSWER HERE

4. **Size versus quality:** Did the larger model in each family (Gemma 3 4B, Llama 3.2 3B) consistently produce better outputs than the smaller one? Were there cases where the size difference did not predict quality?

🖊 TODO: YOUR ANSWER HERE

5. **Runtime trade-offs:** How did elapsed time vary across the four models? Was the quality gain from the slower models worth the additional time, given the nature of this task?

🖊 TODO: YOUR ANSWER HERE

6. **Business risk:** Identify one specific output — from either Part 1 or Part 2 — that would cause a real problem if a person acted on it without review. What is the specific risk, and what kind of human check would catch it?

🖊 TODO: YOUR ANSWER HERE

7. **Generalization:** Based on what you observed, under what conditions would a small local model like these be a reasonable choice for a business task? Under what conditions would you want a larger or cloud-hosted model instead?